<div style="position: relative;" class="flex flex-col"><div class="HTMLContent_html__0OZLp" data-track-load="description_content"><p>You are given a 2D integer <code>grid</code> of size <code>m x n</code> and an integer <code>x</code>. In one operation, you can <strong>add</strong> <code>x</code> to or <strong>subtract</strong> <code>x</code> from any element in the <code>grid</code>.</p>

<p>A <strong>uni-value grid</strong> is a grid where all the elements of it are equal.</p>

<p>Return <em>the <strong>minimum</strong> number of operations to make the grid <strong>uni-value</strong></em>. If it is not possible, return <code>-1</code>.</p>

<p>&nbsp;</p>
<p><strong class="example">Example 1:</strong></p>
<img alt="" style="width: 164px; height: 165px;" src="https://assets.leetcode.com/uploads/2021/09/21/gridtxt.png">
<pre><strong>Input:</strong> grid = [[2,4],[6,8]], x = 2
<strong>Output:</strong> 4
<strong>Explanation:</strong> We can make every element equal to 4 by doing the following: 
- Add x to 2 once.
- Subtract x from 6 once.
- Subtract x from 8 twice.
A total of 4 operations were used.
</pre>

<p><strong class="example">Example 2:</strong></p>
<img alt="" style="width: 164px; height: 165px;" src="https://assets.leetcode.com/uploads/2021/09/21/gridtxt-1.png">
<pre><strong>Input:</strong> grid = [[1,5],[2,3]], x = 1
<strong>Output:</strong> 5
<strong>Explanation:</strong> We can make every element equal to 3.
</pre>

<p><strong class="example">Example 3:</strong></p>
<img alt="" style="width: 164px; height: 165px;" src="https://assets.leetcode.com/uploads/2021/09/21/gridtxt-2.png">
<pre><strong>Input:</strong> grid = [[1,2],[3,4]], x = 2
<strong>Output:</strong> -1
<strong>Explanation:</strong> It is impossible to make every element equal.
</pre>

<p>&nbsp;</p>
<p><strong>Constraints:</strong></p>

<ul>
	<li><code>m == grid.length</code></li>
	<li><code>n == grid[i].length</code></li>
	<li><code>1 &lt;= m, n &lt;= 10<sup>5</sup></code></li>
	<li><code>1 &lt;= m * n &lt;= 10<sup>5</sup></code></li>
	<li><code>1 &lt;= x, grid[i][j] &lt;= 10<sup>4</sup></code></li>
</ul>
</div><span style="font-size: 0px; line-height: 0;">&nbsp;</span></div>

## Initial Intuition

Firstly, we can see that the datastructure does not really matter. For this reason, I think we should put all the elements into an array. This will be useful with later operations.

Next, we should see whether all the items in the grid modulo x result in the same value. If not, it is impossible to make every element equal, as no matter how many x we add or remove from a grid item, if two items do not have the same remainder, they will not be able to be equal.

Then, sorting the array will give the biggest and smallest items. We can sum all the items in the array, and see which of the biggest and smallest multiplied by the length of the array is furthest (i.e. which is furthest from average) to decide which to increment by x, then reinsert into sorted array until all equal the same value (smallest and biggest both equal).

In my first solution, I get lazy and just use list.sort instead of reinserting

In [4]:
from typing import List

In [ ]:
class Solution:
    def minOperations(self, grid: List[List[int]], x: int) -> int:
        row = len(grid[0])
        column = len(grid)
        arr = [0 for _ in range(column * row)]
        for i in range(column):
            for j in range(row):
                arr[(i * row) + j] = grid[i][j]

        lenArr = len(arr)
        firstModulo = arr[0] % x
        sum = arr[0]

        for i in range(1, lenArr):
            curItem = arr[i]

            sum += curItem
            if (curItem % x) != firstModulo:
                return -1
            
        arr.sort()

        count = 0
        while arr[0] != arr[lenArr - 1]:
            count += 1
            if sum - (arr[0] * lenArr) < (arr[lenArr - 1] * lenArr) - sum:
                sum -= x
                arr[lenArr - 1] -= x
            else:
                sum += x
                arr[0] += x

            arr.sort()

        print(arr[0])
        return count

In [ ]:
s = Solution()

s.minOperations(grid = [[1,1, 10000]], x = 1)

## Result

My algorithm exceeds the time limit. This was an expected result, as I have a lot of loops, including having to sort the array everytime. Maybe without this, it would work. However, having to increment by x each time also felt wrong. I could not exactly pinpoint what was wrong, so I decided to ask copilot to why my algorithm was taking long.

Copilot mentioned what I expected, but also mentioned that I should be using median, not average. I applied this to my algorithm to see what happens to my results.

In [ ]:
class Solution1:
    def minOperations(self, grid: List[List[int]], x: int) -> int:
        row = len(grid[0])
        column = len(grid)
        arr = [0 for _ in range(column * row)]
        for i in range(column):
            for j in range(row):
                arr[(i * row) + j] = grid[i][j]

        lenArr = len(arr)
        firstModulo = arr[0] % x
        sum = arr[0]

        for i in range(1, lenArr):
            curItem = arr[i]

            sum += curItem
            if (curItem % x) != firstModulo:
                return -1
            
        
        arr.sort()
        median = arr[lenArr // 2]


        count = 0
        for i in range(lenArr):
            distToMed = abs(arr[i] - median)
            count += distToMed // x

        return count

In [9]:
s = Solution1()

s.minOperations(grid = [[1,1, 10000]], x = 1)

9999

## Result

Success! This solution works in 199ms, beating 32%. Comparing it to leetcodes solution bares similar results. 

### 100%

I looked at the top user solution to see how others managed to get 100%. In Eunice's solution (https://leetcode.com/problems/minimum-operations-to-make-a-uni-value-grid/solutions/8105710/1-by-la_castille-69mz/?envType=daily-question&envId=2026-04-28), we avoid having to sort the array by finding the median by treating the grid as a frequency distribution. So we create a frequency array (similar to counting sort) for all the values after having determined the maximum and minimum (so that we make the smallest possible array). E.g., if x = 4, and the minimum value is 1, maximum is 25, we would only need an array of length 7 to count the frequency. Then, we use cumulative frequency to determine the median. To also save time, instead of checking whether all the numbers share the same modulo in an initial pass, during this cumulative frequency construction, we check the module.